This notebook looks at the distribution of the event records, i.e. a site-event pair is considered as a single datapoint, and realisations are ignored.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pygmt
from pygmt_helper import plotting

import nn_gmm as nng

In [ ]:
imdb_ffp = Path("/Users/claudy/dev/work/data/nn_gmm/20251222_CS200_DSSlab_rotd50_imdb.duckdb")

In [ ]:
with nng.DuckIMDB(imdb_ffp, readonly=True) as imdb:
    site_0_df = imdb.get_site_df(max_grid_level=0, min_grid_level=0)
    sites = site_0_df.site_id.values.astype(str)
    record_info_df = imdb.get_record_info_df(sites=sites)
    site_event_data = imdb.get_site_event_df(sites=site_0_df.site_id.values.astype(str))
    event_df = imdb.get_event_df()

In [ ]:
event_record_info = record_info_df.groupby(["event_int_id", "site_int_id"]).first()

In [ ]:
event_record_info["site_int_id"] = event_record_info.index.get_level_values("site_int_id").values
event_record_info["event_int_id"] = event_record_info.index.get_level_values("event_int_id").values
event_record_info.index = nng.utils.get_site_event_int_id(event_record_info["site_int_id"], event_record_info["event_int_id"])


In [ ]:
event_record_info["mag"] = event_df.loc[event_record_info.event_int_id.values, "magnitude"].values
event_record_info["rrup"] = site_event_data.loc[event_record_info.index.values, "rrup"].values
event_record_info["dtop"] = event_df.loc[event_record_info.event_int_id.values, "dtop"].values
event_record_info["dbottom"] = event_df.loc[event_record_info.event_int_id.values, "dbottom"].values
event_record_info["tect_type"] = event_df.loc[event_record_info.event_int_id.values, "tect_type"].values

# Event Distribution

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 6))

ax1.hist(event_df["magnitude"], bins=25)

ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlabel("Magnitude, $M_W$")
ax1.set_ylabel("Number of records")

# Record Distribution

## Magnitude Distribution

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 6))

ax1.hist(event_record_info["mag"], bins=25)

ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlabel("Magnitude, $M_W$")
ax1.set_ylabel("Number of records")

## Rrup Distribution

In [ ]:
bins = np.logspace(np.log10(1), np.log10(1000), 25)

fig, ax1 = plt.subplots(figsize=(16, 6))

ax1.hist(event_record_info["rrup"], bins=bins)

ax1.set_xlim(1, 1000)
ax1.set_xscale("log")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlabel("Source-to-site distance, $R_{Rup}$ (km)")
ax1.set_ylabel("Number of records")

## Magnitude - Rrup Scatter

In [ ]:
figsize = (16, 8)
n_rrup_bins = 20
n_mag_bins = 20

fig, axs = plt.subplot_mosaic(
    [["histx", "."], ["scatter", "histy"]],
    width_ratios=(4, 1),
    height_ratios=(1, 4),
    layout="constrained",
    figsize=figsize,
)
ax_scatter = axs["scatter"]
ax_histx = axs["histx"]
ax_histy = axs["histy"]

# Scatter plot
ax_scatter.scatter(
    event_record_info["rrup"],
    event_record_info["mag"],
    s=2.5,
    c="r",
    alpha=0.5,
)

ax_scatter.set_xlabel("Source-to-site distance, $R_{Rup}$ (km)")
ax_scatter.set_ylabel("Magnitude, $M_{W}$")
ax_scatter.set_xscale("log")
ax_scatter.set_xlim(0.1, 1000)
ax_scatter.set_ylim(4.0, None)
ax_scatter.grid(which="both", linewidth=0.5, alpha=0.5, linestyle="--")

ax_histx.hist(
    event_record_info["rrup"],
    bins=np.logspace(np.log10(0.1), np.log10(1000), n_rrup_bins),
    color="red",
)
ax_histx.set_xscale("log")
ax_histx.set_xlim(0.1, 1000)
ax_histx.spines[["top", "right"]].set_visible(False)
ax_histx.set_xticklabels([])

ax_histy.hist(
    event_record_info["mag"],
    bins=n_mag_bins,
    color="red",
    orientation="horizontal",
)
ax_histy.set_ylim(ax_scatter.get_ylim())
ax_histy.spines[["top", "right"]].set_visible(False)
ax_histy.set_yticklabels([]);

## DTop Distribution

In [ ]:
# bins = np.logspace(np.log10(1), np.log10(1000), 25)

fig, ax1 = plt.subplots(figsize=(16, 6))

# ax1.hist(event_record_info["dtop"])
sns.histplot(event_record_info, x="dtop", hue="tect_type", bins=25, ax=ax1)

ax1.set_xlim(1, 1000)
ax1.set_xscale("log")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlabel("Source-to-site distance, $R_{Rup}$ (km)")
ax1.set_ylabel("Number of records")